In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.layers import Input, Embedding, Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l2
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

print("TensorFlow Version:", tf.__version__)
print("GPU Available:", len(tf.config.list_physical_devices('GPU')) > 0)

In [ ]:
!pip install -q datasets

from datasets import load_dataset

dataset = load_dataset("go_emotions", "simplified")

df_train = pd.DataFrame(dataset['train'])
df_val = pd.DataFrame(dataset['validation'])
df_test = pd.DataFrame(dataset['test'])

# Combine train+val for better training
df = pd.concat([df_train, df_val], ignore_index=True)

print(f"Train+Val Shape: {df.shape}")
print(f"Test Shape: {df_test.shape}")
print(f"Columns: {df.columns.tolist()}")

In [ ]:
emotion_labels_map = {
    0: 'admiration', 1: 'amusement', 2: 'anger', 3: 'annoyance', 4: 'approval',
    5: 'caring', 6: 'confusion', 7: 'curiosity', 8: 'desire', 9: 'disappointment',
    10: 'disapproval', 11: 'disgust', 12: 'embarrassment', 13: 'excitement', 14: 'fear',
    15: 'gratitude', 16: 'grief', 17: 'joy', 18: 'love', 19: 'nervousness',
    20: 'optimism', 21: 'pride', 22: 'realization', 23: 'relief', 24: 'remorse',
    25: 'sadness', 26: 'surprise', 27: 'neutral'
}

id_to_emotion = emotion_labels_map
emotion_to_id = {v: k for k, v in emotion_labels_map.items()}

NUM_CLASSES = 28
print(f"Total Classes: {NUM_CLASSES}")
for idx, name in emotion_labels_map.items():
    print(f"  {idx}: {name}")

In [ ]:
from collections import Counter
def get_primary_label(label_list):
    if len(label_list) > 0:
        return label_list[0]
    return 27  # neutral default

df['primary_label'] = df['labels'].apply(get_primary_label)
df_test['primary_label'] = df_test['labels'].apply(get_primary_label)

print("=== Before Balancing ===")
label_counts = Counter(df['primary_label'].tolist())
for idx, count in sorted(label_counts.items()):
    print(f"  {emotion_labels_map[idx]}: {count}")

# Plot
plt.figure(figsize=(14, 5))
names = [emotion_labels_map[i] for i in sorted(label_counts.keys())]
counts = [label_counts[i] for i in sorted(label_counts.keys())]
plt.bar(names, counts, color='steelblue')
plt.xticks(rotation=90)
plt.title('Before Balancing — GoEmotions')
plt.tight_layout()
plt.show()

In [ ]:
#data cleaning

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['cleaned_text'] = df['text'].apply(clean_text)
df_test['cleaned_text'] = df_test['text'].apply(clean_text)

# Remove empty
df = df[df['cleaned_text'] != ""].reset_index(drop=True)
df_test = df_test[df_test['cleaned_text'] != ""].reset_index(drop=True)

print(f"Train+Val after cleaning: {df.shape}")
print(f"Test after cleaning: {df_test.shape}")

print("\n=== Sample ===")
for i in range(3):
    print(f"Original : {df['text'].iloc[i]}")
    print(f"Cleaned  : {df['cleaned_text'].iloc[i]}")
    print(f"Emotion  : {emotion_labels_map[df['primary_label'].iloc[i]]}")
    print("-" * 50)

In [ ]:
MAX_SAMPLES = 2500   # Majority class cap
MIN_SAMPLES = 400    # Minority class minimum

texts_raw = df['cleaned_text'].tolist()
labels_raw = df['primary_label'].tolist()

balanced_texts = []
balanced_labels = []

np.random.seed(42)

for emotion_id in range(NUM_CLASSES):
    # Is class ke saare indices
    indices = [i for i, y in enumerate(labels_raw) if y == emotion_id]

    if len(indices) == 0:
        print(f"Warning: {emotion_labels_map[emotion_id]} has 0 samples!")
        continue

    if len(indices) > MAX_SAMPLES:
        # Undersample — zyada wale ko kam karo
        indices = np.random.choice(indices, MAX_SAMPLES, replace=False).tolist()
    elif len(indices) < MIN_SAMPLES:
        # Oversample — kam wale ko boost karo
        indices = np.random.choice(indices, MIN_SAMPLES, replace=True).tolist()

    balanced_texts.extend([texts_raw[i] for i in indices])
    balanced_labels.extend([emotion_id] * len(indices))

# Shuffle
combined = list(zip(balanced_texts, balanced_labels))
np.random.shuffle(combined)
balanced_texts, balanced_labels = zip(*combined)
balanced_texts = list(balanced_texts)
balanced_labels = list(balanced_labels)

print(f"Balanced Dataset Size: {len(balanced_texts)}")
print("\n=== After Balancing ===")
balanced_counts = Counter(balanced_labels)
for idx in sorted(balanced_counts.keys()):
    print(f"  {emotion_labels_map[idx]}: {balanced_counts[idx]}")

# Plot
plt.figure(figsize=(14, 5))
names = [emotion_labels_map[i] for i in sorted(balanced_counts.keys())]
counts = [balanced_counts[i] for i in sorted(balanced_counts.keys())]
plt.bar(names, counts, color='seagreen')
plt.xticks(rotation=90)
plt.title('After Balancing — GoEmotions')
plt.tight_layout()
plt.show()

In [ ]:
MAX_VOCAB_SIZE = 20000
MAX_SEQUENCE_LENGTH = 50

tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts(balanced_texts)

word_index = tokenizer.word_index
vocab_size = min(len(word_index) + 1, MAX_VOCAB_SIZE)
print(f"Vocabulary Size: {vocab_size}")

# Sequences + Padding
X_balanced = pad_sequences(
    tokenizer.texts_to_sequences(balanced_texts),
    maxlen=MAX_SEQUENCE_LENGTH, padding='post', truncating='post'
)
y_balanced = np.array(balanced_labels)

# Test set
X_test = pad_sequences(
    tokenizer.texts_to_sequences(df_test['cleaned_text'].tolist()),
    maxlen=MAX_SEQUENCE_LENGTH, padding='post', truncating='post'
)
y_test = df_test['primary_label'].values

print(f"X_balanced shape: {X_balanced.shape}")
print(f"X_test shape: {X_test.shape}")

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X_balanced, y_balanced,
    test_size=0.15,
    random_state=42,
    stratify=y_balanced
)

# One-hot encode
y_train_cat = to_categorical(y_train, num_classes=NUM_CLASSES)
y_val_cat = to_categorical(y_val, num_classes=NUM_CLASSES)
y_test_cat = to_categorical(y_test, num_classes=NUM_CLASSES)

print(f"X_train: {X_train.shape}")
print(f"X_val:   {X_val.shape}")
print(f"X_test:  {X_test.shape}")

In [ ]:
# class wieght

from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weight_dict = dict(enumerate(class_weights))

print("Class Weights:")
for idx, weight in class_weight_dict.items():
    print(f"  {emotion_labels_map[idx]}: {weight:.3f}")

In [ ]:
BATCH_SIZE = 64

train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train_cat))
train_dataset = train_dataset.shuffle(10000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_dataset = tf.data.Dataset.from_tensor_slices((X_val, y_val_cat))
val_dataset = val_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

test_dataset = tf.data.Dataset.from_tensor_slices((X_test, y_test_cat))
test_dataset = test_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print(f"Train Batches: {len(list(train_dataset))}")
print(f"Val Batches:   {len(list(val_dataset))}")
print(f"Test Batches:  {len(list(test_dataset))}")

In [ ]:
def create_bilstm_model(vocab_size, max_length, embedding_dim=128,
                         lstm_units=128, num_classes=28, dropout_rate=0.3):
    inputs = Input(shape=(max_length,))

    # Embedding
    x = Embedding(vocab_size, embedding_dim,
                  input_length=max_length, mask_zero=True)(inputs)
    x = Dropout(0.2)(x)

    # BiLSTM Layer 1
    x = Bidirectional(LSTM(lstm_units,
                           return_sequences=True,
                           dropout=0.3,
                           recurrent_dropout=0.2))(x)

    # BiLSTM Layer 2
    x = Bidirectional(LSTM(64,
                           return_sequences=False,
                           dropout=0.3,
                           recurrent_dropout=0.2))(x)

    # Dense layers
    x = Dense(256, activation='relu', kernel_regularizer=l2(0.001))(x)
    x = Dropout(dropout_rate)(x)
    x = Dense(128, activation='relu', kernel_regularizer=l2(0.001))(x)
    x = Dropout(dropout_rate)(x)

    outputs = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs, outputs, name='BiLSTM_GoEmotions_Balanced')
    return model

model = create_bilstm_model(vocab_size, MAX_SEQUENCE_LENGTH)
model.summary()

In [ ]:
loss_fn = tf.keras.losses.CategoricalCrossentropy()
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

# LR Scheduler
lr_scheduler = tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=0.001,
    decay_steps=1000,
    decay_rate=0.9
)
optimizer = tf.keras.optimizers.Adam(learning_rate=lr_scheduler)

train_loss_m = tf.keras.metrics.Mean()
train_acc_m = tf.keras.metrics.CategoricalAccuracy()
val_loss_m = tf.keras.metrics.Mean()
val_acc_m = tf.keras.metrics.CategoricalAccuracy()

@tf.function
def train_step(x, y, weights):
    with tf.GradientTape() as tape:
        pred = model(x, training=True)
        loss = loss_fn(y, pred)
        # Class weights apply karo
        sample_weights = tf.reduce_sum(
            tf.cast(y, tf.float32) *
            tf.constant(list(class_weight_dict.values()), dtype=tf.float32),
            axis=1
        )
        loss = tf.reduce_mean(loss * sample_weights)
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    train_loss_m.update_state(loss)
    train_acc_m.update_state(y, pred)

@tf.function
def val_step(x, y):
    pred = model(x, training=False)
    loss = loss_fn(y, pred)
    val_loss_m.update_state(loss)
    val_acc_m.update_state(y, pred)

EPOCHS = 20
best_val_acc = 0
patience_counter = 0
PATIENCE = 4
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

print("Training Started...")
print("=" * 65)

for epoch in range(EPOCHS):
    train_loss_m.reset_state()
    train_acc_m.reset_state()
    val_loss_m.reset_state()
    val_acc_m.reset_state()

    for x_batch, y_batch in train_dataset:
        train_step(x_batch, y_batch, class_weight_dict)

    for x_batch, y_batch in val_dataset:
        val_step(x_batch, y_batch)

    tl = train_loss_m.result().numpy()
    ta = train_acc_m.result().numpy()
    vl = val_loss_m.result().numpy()
    va = val_acc_m.result().numpy()

    history['train_loss'].append(tl)
    history['train_acc'].append(ta)
    history['val_loss'].append(vl)
    history['val_acc'].append(va)

    print(f"Epoch {epoch+1:02d}/{EPOCHS} | "
          f"Train Loss: {tl:.4f} Acc: {ta:.4f} | "
          f"Val Loss: {vl:.4f} Acc: {va:.4f}")

    if va > best_val_acc:
        best_val_acc = va
        model.save_weights('best_bilstm_goemotion_balanced.weights.h5')
        print(f"  ✓ Saved! Best Val Acc: {best_val_acc:.4f}")
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"  Early Stopping at Epoch {epoch+1}")
            break

print(f"\nBest Val Accuracy: {best_val_acc:.4f}")

In [ ]:
import seaborn as sns

model.load_weights('best_bilstm_goemotion_balanced.weights.h5')

y_pred = np.argmax(model.predict(X_test), axis=1)
y_true = y_test

emotion_names = [emotion_labels_map[i] for i in range(NUM_CLASSES)]
print(classification_report(y_true, y_pred, target_names=emotion_names))

# Confusion Matrix
plt.figure(figsize=(16, 14))
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d',
            xticklabels=emotion_names,
            yticklabels=emotion_names,
            cmap='Blues')
plt.title('Confusion Matrix — GoEmotions BiLSTM Balanced')
plt.ylabel('True')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

# Training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(history['train_loss'], label='Train')
ax1.plot(history['val_loss'], label='Val')
ax1.set_title('Loss')
ax1.legend()

ax2.plot(history['train_acc'], label='Train')
ax2.plot(history['val_acc'], label='Val')
ax2.set_title('Accuracy')
ax2.legend()
plt.tight_layout()
plt.show()

In [ ]:
import pickle

model.save('bilstm_goemotion_balanced.h5')
print("✓ Model Saved")

with open('goemotion_tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)
print("✓ Tokenizer Saved")

with open('goemotion_label_mapping.pkl', 'wb') as f:
    pickle.dump(emotion_labels_map, f)
print("✓ Label Mapping Saved")

with open('goemotion_max_length.pkl', 'wb') as f:
    pickle.dump(MAX_SEQUENCE_LENGTH, f)
print("✓ Max Length Saved")

In [ ]:
def predict_emotion(text):
    cleaned = clean_text(text)
    seq = tokenizer.texts_to_sequences([cleaned])
    padded = pad_sequences(seq, maxlen=MAX_SEQUENCE_LENGTH,
                          padding='post', truncating='post')
    pred = model.predict(padded, verbose=0)
    emotion_idx = np.argmax(pred)
    confidence = pred[0][emotion_idx]

    # Top 3 emotions
    top3_idx = np.argsort(pred[0])[::-1][:3]
    print(f"Text: {text}")
    print(f"Top Emotions:")
    for idx in top3_idx:
        print(f"  {emotion_labels_map[idx]}: {pred[0][idx]*100:.1f}%")
    return emotion_labels_map[emotion_idx], confidence

# Test
sentences = [
    "I am feeling very anxious and cannot sleep",
    "I am so happy and excited today!",
    "I hate everything, nothing is going right",
    "I feel so alone and nobody understands me",
    "Thank you so much, you really helped me"
]

print("=" * 55)
for s in sentences:
    emotion, conf = predict_emotion(s)
    print(f"→ Predicted: {emotion} ({conf*100:.1f}%)")
    print("-" * 55)

In [ ]:
# Install Transformers
!pip install -q transformers datasets accelerate

In [ ]:
from transformers import BlenderbotTokenizer, BlenderbotForConditionalGeneration
import torch
import pandas as pd
import re
import numpy as np
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset

In [ ]:
# 1. Model & Tokenizer Fresh Load
# ============================================
model_name = "facebook/blenderbot-400M-distill"
tokenizer = BlenderbotTokenizer.from_pretrained(model_name)
model = BlenderbotForConditionalGeneration.from_pretrained(
    model_name,
    torch_dtype=torch.float32
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print(f"Model loaded on {device}")

In [ ]:
# Load MentalChat16K Dataset
from datasets import load_dataset
import pandas as pd

dataset = load_dataset("ShenLab/MentalChat16K")
df = pd.DataFrame(dataset['train'])
print(f"MentalChat16K Loaded! Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head()

In [ ]:
# 2. Data Clean
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_input'] = df['input'].apply(clean_text)
df['clean_output'] = df['output'].apply(clean_text)
df = df[(df['clean_input'] != "") & (df['clean_output'] != "")]
df = df.reset_index(drop=True)
print(f"Dataset size: {df.shape}")

In [ ]:
# 3. Tokenization —
MAX_LENGTH = 128

class MentalChatDataset(Dataset):
    def __init__(self, inputs, outputs, tokenizer, max_length):
        self.inputs = inputs
        self.outputs = outputs
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        # Input tokenize
        inp = self.tokenizer(
            self.inputs[idx],
            truncation=True,
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt'
        )
        # Output tokenize
        out = self.tokenizer(
            self.outputs[idx],
            truncation=True,
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt'
        )

        labels = out['input_ids'].squeeze()

        # Padding tokens
        labels[labels == self.tokenizer.pad_token_id] = -100

        return {
            'input_ids': inp['input_ids'].squeeze(),
            'attention_mask': inp['attention_mask'].squeeze(),
            'labels': labels
        }

In [ ]:
# 4. Train-Val Split
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['clean_input'].tolist(),
    df['clean_output'].tolist(),
    test_size=0.1,
    random_state=42
)

train_dataset = MentalChatDataset(train_texts, train_labels, tokenizer, MAX_LENGTH)
val_dataset = MentalChatDataset(val_texts, val_labels, tokenizer, MAX_LENGTH)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")

In [ ]:
# 5. Verify — Loss Check
# ============================================
sample = train_dataset[0]
input_ids = sample['input_ids'].unsqueeze(0).to(device)
attention_mask = sample['attention_mask'].unsqueeze(0).to(device)
labels = sample['labels'].unsqueeze(0).to(device)

print(f"Non -100 labels: {(labels != -100).sum()}")

with torch.no_grad():
    outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
    print(f"Loss: {outputs.loss}")

In [ ]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback

training_args = TrainingArguments(
    output_dir='./blenderbot-mental-health',
    num_train_epochs=5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    warmup_steps=100,
    weight_decay=0.01,
    logging_steps=100,
    eval_strategy='steps',
    eval_steps=500,
    save_steps=1000,
    save_total_limit=2,
    load_best_model_at_end=True,
    fp16=True,
    report_to='none',
    learning_rate=5e-5,
    remove_unused_columns=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer.train()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

steps = [500, 1000, 1500, 2000, 2500, 3000, 3500, 4000, 4500]
val_loss = [1.2667, 1.1577, 1.1046, 1.0839, 1.0621, 1.0705, 1.0564, 1.0691, 1.0645]
train_loss = [5.388, 4.319, 4.209, 3.631, 3.564, 3.175, 3.154, 2.882, 2.863]

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(steps, train_loss, marker='o', label='Train Loss', color='steelblue')
plt.plot(steps, val_loss, marker='o', label='Val Loss', color='orange')
plt.axvline(x=3500, color='red', linestyle='--', label='Best Model')
plt.xlabel('Steps')
plt.ylabel('Loss')
plt.title('BlenderBot Training vs Validation Loss')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(steps, val_loss, marker='o', color='orange')
plt.axvline(x=3500, color='red', linestyle='--', label='Best (Step 3500)')
plt.fill_between(steps, val_loss, alpha=0.2, color='orange')
plt.xlabel('Steps')
plt.ylabel('Validation Loss')
plt.title('Validation Loss Progress')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig('blenderbot_loss_plot.png', dpi=150)
plt.show()


In [ ]:
import torch
from transformers import BlenderbotTokenizer, BlenderbotForConditionalGeneration

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_path = './blenderbot-mental-health/checkpoint-4000'
tokenizer = BlenderbotTokenizer.from_pretrained(model_path)
model = BlenderbotForConditionalGeneration.from_pretrained(
    model_path,
    torch_dtype=torch.float32
)
model = model.to(device)
model.eval()

def generate_response(user_input):
    inputs = tokenizer(
        user_input,
        return_tensors='pt',
        truncation=True,
        max_length=128
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            min_new_tokens=10,
            num_beams=4,
            no_repeat_ngram_size=3,
            early_stopping=True
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response

# Test karo
test_inputs = [

    "I am angry at everything"
]

for text in test_inputs:
    print(f"User: {text}")
    print(f"Bot : {generate_response(text)}")
    print("-" * 50)

In [ ]:
from transformers import BlenderbotTokenizer, BlenderbotForConditionalGeneration
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Model checkpoint se load karo
# Lekin tokenizer original facebook se load karo
tokenizer = BlenderbotTokenizer.from_pretrained('facebook/blenderbot-400M-distill')
model = BlenderbotForConditionalGeneration.from_pretrained(
    './blenderbot-mental-health/checkpoint-4000',
    torch_dtype=torch.float32
)
model = model.to(device)
model.eval()

# Debug
text = "I am feeling very sad and alone today"
inputs = tokenizer(
    text,
    return_tensors='pt',
    truncation=True,
    max_length=128
).to(device)

print(f"Input IDs: {inputs['input_ids']}")
print(f"Input decoded: {tokenizer.decode(inputs['input_ids'][0])}")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        min_new_tokens=10,
        num_beams=4,
        no_repeat_ngram_size=3,
    )

print(f"Response: {tokenizer.decode(outputs[0], skip_special_tokens=True)}")

In [ ]:
def generate_response(user_input):
    inputs = tokenizer(
        user_input,
        return_tensors='pt',
        truncation=True,
        max_length=128
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            min_new_tokens=20,
            num_beams=4,
            no_repeat_ngram_size=3,
            early_stopping=True
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

test_inputs = [
    "I lost my job today and I feel devastated",
    "I feel so anxious and cannot sleep at night",
    "I am angry at everything around me",
    "I feel hopeless and do not know what to do",
    "I feel scared about my future",
    "Nobody understands me and I feel very alone"
]

print("MENTAL HEALTH CHATBOT TEST")
print("=" * 60)
for text in test_inputs:
    print(f"User : {text}")
    print(f"Bot  : {generate_response(text)}")
    print("-" * 60)

In [ ]:
# Test Model Completely Offline (No HuggingFace)
from transformers import BlenderbotTokenizer, BlenderbotForConditionalGeneration

model_path = './blenderbot-mental-health/checkpoint-4000'
tokenizer = BlenderbotTokenizer.from_pretrained(model_path)
model = BlenderbotForConditionalGeneration.from_pretrained(model_path)

def generate_response(user_input, emotion):
    prompt = f"User feeling {emotion}: {user_input}"
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=128)
    outputs = model.generate(
        inputs.input_ids,
        max_new_tokens=100,
        min_new_tokens=5,
        num_beams=4,
        no_repeat_ngram_size=3
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

tests = [
    ("i lost my job and i feel worthless", "sadness"),
    ("i am so grateful for my friends support", "gratitude"),
    ("i feel so nervous before every social interaction", "nervousness")
]

for user_input, emotion in tests:
    print(f"Emotion: {emotion}")
    print(f"User: {user_input}")
    print(f"Bot: {generate_response(user_input, emotion)}")
    print("-" * 50)